# 专题: 一维与二维水力模型对比

## 目的
本项目框架中同时包含了一维（`preissmann_model`）和二维（`model_2d`）水力模型。这两种模型在模拟河道水流时各有优劣。此文档旨在通过一个相同的、简化的算例，对这两种模型进行直接的对比。

此笔记本将：
1.  建立一个简单的长直矩形渠道场景。
2.  分别使用`preissmann_model`（1D）和`model_2d`（2D）对这个渠道进行洪水波演进模拟。
3.  将两个模型的计算结果（如沿程水位、出口流量过程线）绘制在一起进行比较。
4.  讨论一维和二维模型在结果、计算成本和数据需求方面的差异，以帮助用户根据实际需求选择合适的模型。

## 1. 环境设置

我们导入两个模型所需的所有组件。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import time

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

# 1D模型组件
from preissmann_model.model import HydraulicModel as Model1D
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection

# 2D模型组件
from model_2d.model import Model2D
from model_2d.mesh import Mesh

## 2. 定义共享的场景参数

我们定义一个长1000米、宽100米的矩形渠道。我们将使用一个相同的三角形入流过程线来模拟洪水波。

In [ ]:
# 渠道几何
channel_length = 1000.0 # m
channel_width = 100.0  # m
channel_slope = 0.001  # m/m
manning_n = 0.03

# 模拟参数
num_steps = 150
dt_1d = 10 # s
dt_2d = 1 # s (2D模型通常需要更小的时间步)

# 入流过程线
q_inflow = np.array([10] * 10 + [10, 20, 30, 50, 80, 100, 70, 50, 30, 20] + [10] * (num_steps - 20))
downstream_z = 2.0 # m

## 3. 运行一维模型 (Preissmann)


In [ ]:
print("--- 运行1D Preissmann模型 ---")
num_nodes_1d = 21
reach_1d = RiverReach(
    cross_sections=[RectangularCrossSection(width=channel_width) for _ in range(num_nodes_1d)],
    lengths=np.full(num_nodes_1d - 1, channel_length / (num_nodes_1d - 1)),
    slope=channel_slope,
    manning_n=manning_n
)
model_1d = Model1D(name="model_1d", reach=reach_1d, dt=dt_1d, downstream_level=downstream_z)
model_1d.Q[:] = q_inflow[0]
model_1d.Z[:] = downstream_z # 初始为平坦水面

q_out_1d = []
start_time_1d = time.time()
for i in range(num_steps):
    model_1d.step({'Q_inflow': q_inflow[i]}, dt_1d)
    q_out_1d.append(model_1d.get_outflow())
end_time_1d = time.time()
print(f"1D模型运行完成，耗时: {end_time_1d - start_time_1d:.4f} 秒")

## 4. 运行二维模型

为了模拟同一个渠道，我们需要为2D模型创建一个代表该渠道的三角网格。

In [ ]:
print("\n--- 运行2D 有限体积模型 ---")
# 创建2D网格 (50x5的矩形网格)
nx, ny = 51, 6
x_coords = np.linspace(0, channel_length, nx)
y_coords = np.linspace(-channel_width/2, channel_width/2, ny)
points_2d = [(x, y) for x in x_coords for y in y_coords]
triangles_2d = []
for i in range(nx - 1):
    for j in range(ny - 1):
        p0 = i * ny + j
        p1 = (i + 1) * ny + j
        p2 = (i + 1) * ny + j + 1
        p3 = i * ny + j + 1
        triangles_2d.append((p0, p1, p2))
        triangles_2d.append((p0, p2, p3))

mesh_2d = Mesh()
mesh_2d.build_from_points_and_triangles(points_2d, triangles_2d)

model_2d = Model2D(name="model_2d", mesh=mesh_2d)
# 设置初始和边界条件
for face in model_2d.mesh.faces:
    face.z_bed = (channel_length - face.centroid[0]) * channel_slope
    face.h = max(0, downstream_z - face.z_bed)
model_2d.set_inflow_boundary(q_inflow[0], 0)

q_out_2d = []
start_time_2d = time.time()
for i in range(num_steps):
    model_2d.set_inflow_boundary(q_inflow[i], 0)
    model_2d.step({}, dt_2d)
    q_out_2d.append(model_2d.get_outflow())
end_time_2d = time.time()
print(f"2D模型运行完成，耗时: {end_time_2d - start_time_2d:.4f} 秒")

## 5. 结果对比

### 5.1 出口流量过程线对比

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
timesteps_1d = np.arange(num_steps) * dt_1d / 60 # minutes
timesteps_2d = np.arange(num_steps) * dt_2d / 60 # minutes

ax.plot(timesteps_1d, q_inflow, 'k--', label='Inflow Hydrograph')
ax.plot(timesteps_1d, q_out_1d, 'b-o', label='1D Model Outflow')
ax.plot(timesteps_2d, q_out_2d, 'r-s', label='2D Model Outflow')

ax.set_title('1D vs 2D Model: Outlet Hydrograph Comparison')
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Discharge (m³/s)')
ax.legend()
ax.grid(True)
plt.show()

### 5.2 最终水面线对比

我们提取2D模型中线上的水位，与1D模型的水位进行比较。

In [ ]:
# 提取2D模型中线水位
z_2d_centerline = []
dist_2d_centerline = []
for face in model_2d.mesh.faces:
    if abs(face.centroid[1]) < 1.0: # 假设中线y=0附近
        z_2d_centerline.append(face.h + face.z_bed)
        dist_2d_centerline.append(face.centroid[0])
# 排序
sorted_indices = np.argsort(dist_2d_centerline)
dist_2d_centerline = np.array(dist_2d_centerline)[sorted_indices]
z_2d_centerline = np.array(z_2d_centerline)[sorted_indices]

# 1D模型距离
dist_1d = np.cumsum(np.insert(model_1d.reach.lengths, 0, 0))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(dist_1d, model_1d.Z, 'b-o', label='1D Model WSE')
ax.plot(dist_2d_centerline, z_2d_centerline, 'r-s', label='2D Model WSE (Centerline)')
ax.plot(dist_1d, model_1d.Z_bed, 'k-', label='Bed Elevation')

ax.set_title('1D vs 2D Model: Final Water Surface Elevation')
ax.set_xlabel('Distance from Upstream (m)')
ax.set_ylabel('Elevation (m)')
ax.legend()
ax.grid(True)
plt.show()

## 6. 讨论

- **结果相似性**: 尽管两个模型的数值方法和维度完全不同，但它们都很好地模拟了洪水波的传播，包括洪峰的削减和延迟，最终得到的出口过程线和水面线趋势非常相似。这验证了两个模型在模拟简单一维流动时的正确性。
- **计算成本**: 从打印出的运行时间可以看到，**2D模型的计算成本远高于1D模型**。这主要是因为2D模型需要在更多的计算单元上进行计算，并且为了维持数值稳定性，通常需要更小的时间步长。
- **数据需求和适用性**:
    - **1D模型**: 需要的数据量少（断面形状、间距、糙率），计算速度快。非常适用于模拟长河段的、流动方向基本单一的河道洪水演进。
    - **2D模型**: 需要详细的二维地形数据（DEM），计算量大。但它能够模拟复杂的二维流场，如漫滩、分汇流、弯道水流等。当水流不仅限于单一方向时，2D模型是必需的。

**结论**: 在选择模型时，应根据研究目的和区域特点进行权衡。对于干流洪水演进等一维特征明显的场景，1D模型是高效且准确的选择。对于需要精细模拟局部二维水流现象（如洪水淹没范围）的场景，则应使用2D模型。